# Observability: Evaluation Pipeline

----

This notebook demonstrates a **stage-by-stage evaluation pipeline** for a Semantic Kernel-based chat application, using Application Insights logs and the **Azure AI Evaluation SDK (MS Foundry)**.

You will learn how to:

- **Configure Application Insights** telemetry for Semantic Kernel
- **Build an SK orchestration layer**: intent classification → agent routing → function calling → answer generation
- **Parse Application Insights logs** into structured evaluation datasets
- **Run custom evaluators**: intent accuracy (exact match), agent relevance, method relevance
- **Run pre-built evaluators**: similarity, groundedness via Azure AI Evaluation SDK
- **Aggregate results** into a unified evaluation dashboard

## Table of Contents

- [Setup](#setup)
- [Part 1: Application Insights + Semantic Kernel Telemetry](#part-1-application-insights--semantic-kernel-telemetry)
- [Part 2: Semantic Kernel Orchestration Layer](#part-2-semantic-kernel-orchestration-layer)
- [Part 3: Build Evaluation Dataset from App Insights Logs](#part-3-build-evaluation-dataset-from-app-insights-logs)
- [Part 4: Evaluation Step 1 – Intent Classification (Exact Match)](#part-4-evaluation-step-1--intent-classification-exact-match)
- [Part 5: Evaluation Step 2 – Agent Relevance](#part-5-evaluation-step-2--agent-relevance)
- [Part 6: Evaluation Step 3 – Method Relevance](#part-6-evaluation-step-3--method-relevance)
- [Part 7: Evaluation Step 4 – Similarity & Groundedness (Pre-built)](#part-7-evaluation-step-4--similarity--groundedness-pre-built)
- [Part 8: Evaluation Summary Dashboard](#part-8-evaluation-summary-dashboard)
- [Wrap-up](#wrap-up)

## Setup

This notebook reuses the configuration file (`.foundry_config.json`) created by `0_setup/1_setup.ipynb`.

### Prerequisites

| Variable | Description |
|----------|-------------|
| `APPLICATIONINSIGHTS_CONNECTION_STRING` | Azure Application Insights connection string |
| `AZURE_OPENAI_ENDPOINT` | Azure OpenAI endpoint URL |
| `AZURE_OPENAI_API_KEY` | Azure OpenAI API key |
| `AZURE_OPENAI_CHAT_DEPLOYMENT_NAME` | Deployment name (e.g., `gpt-4.1`) |
| `AZURE_AI_PROJECT_ENDPOINT` | Azure AI Foundry project endpoint |

In [ ]:
# Environment setup and PATH configuration
import json
import os
import subprocess
import csv
from typing import Dict, List, Any
from dataclasses import dataclass
from dotenv import load_dotenv

load_dotenv(override=True)

# Ensure the notebook kernel can find Azure CLI on PATH
possible_paths = [
    '/opt/homebrew/bin',
    '/usr/local/bin',
    '/usr/bin',
    '/home/linuxbrew/.linuxbrew/bin',
]

az_path = None
try:
    result = subprocess.run(['which', 'az'], capture_output=True, text=True)
    if result.returncode == 0:
        az_path = os.path.dirname(result.stdout.strip())
        print(f'🔍 Azure CLI found: {result.stdout.strip()}')
except Exception:
    pass

paths_to_add: list[str] = []
if az_path and az_path not in os.environ.get('PATH', ''):
    paths_to_add.append(az_path)
else:
    for path in possible_paths:
        if os.path.exists(path) and path not in os.environ.get('PATH', ''):
            paths_to_add.append(path)

if paths_to_add:
    os.environ['PATH'] = ':'.join(paths_to_add) + ':' + os.environ.get('PATH', '')
    print(f"✅ Added to PATH: {', '.join(paths_to_add)}")
else:
    print('✅ PATH looks good already')

In [ ]:
# Load Foundry project settings
from azure.identity import DefaultAzureCredential

config_file = '../0_setup/.foundry_config.json'
try:
    with open(config_file, 'r', encoding='utf-8') as f:
        config = json.load(f)
except FileNotFoundError as e:
    print(f"⚠️ Could not find '{config_file}'.")
    print('💡 Run 0_setup/1_setup.ipynb first to create it.')
    raise e

FOUNDRY_NAME = config.get('FOUNDRY_NAME')
RESOURCE_GROUP = config.get('RESOURCE_GROUP')
LOCATION = config.get('LOCATION')
PROJECT_NAME = config.get('PROJECT_NAME', 'proj-default')
AZURE_AI_PROJECT_ENDPOINT = config.get('AZURE_AI_PROJECT_ENDPOINT')

AZURE_OPENAI_ENDPOINT = os.environ.get("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_KEY = os.environ.get("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_CHAT_DEPLOYMENT_NAME = os.environ.get(
    "AZURE_OPENAI_CHAT_DEPLOYMENT_NAME", "gpt-4.1"
)
AZURE_OPENAI_API_VERSION = os.environ.get(
    "AZURE_OPENAI_API_VERSION", "2025-03-01-preview"
)
CONNECTION_STRING = os.environ.get("APPLICATIONINSIGHTS_CONNECTION_STRING")

os.environ['FOUNDRY_NAME'] = FOUNDRY_NAME or ''
os.environ['LOCATION'] = LOCATION or ''
os.environ['RESOURCE_GROUP'] = RESOURCE_GROUP or ''
os.environ['AZURE_SUBSCRIPTION_ID'] = config.get('AZURE_SUBSCRIPTION_ID', '')

credential = DefaultAzureCredential()

print(f"✅ Loaded settings from '{config_file}'.")
print(f"📌 Foundry: {FOUNDRY_NAME} | Region: {LOCATION}")
print(f"📌 Project endpoint: {AZURE_AI_PROJECT_ENDPOINT}")
print(f"📌 Chat deployment: {AZURE_OPENAI_CHAT_DEPLOYMENT_NAME}")

## Part 1: Application Insights + Semantic Kernel Telemetry

Semantic Kernel은 OpenTelemetry를 기본 지원하며, `azure-monitor-opentelemetry`를 통해 Application Insights로 모든 트레이스를 자동 전송할 수 있습니다.

아래 설정으로 SK의 LLM 호출, 플러그인 실행, function calling 등이 모두 Application Insights에 자동 기록됩니다.

```
┌─────────────────────────────────────────────────┐
│            Semantic Kernel App                   │
│                                                  │
│  User Query → Intent Classification              │
│            → Agent Routing (SK Plugin)            │
│            → Function Calling (auto)              │
│            → Final Answer Generation              │
│                                                  │
│  OpenTelemetry SDK ──► Azure Monitor Exporter     │
│                        ──► Application Insights   │
└─────────────────────────────────────────────────┘
```

In [ ]:
# Configure OpenTelemetry + Azure Monitor for Semantic Kernel
from opentelemetry import trace
from azure.monitor.opentelemetry import configure_azure_monitor

if not CONNECTION_STRING:
    raise ValueError(
        "❌ APPLICATIONINSIGHTS_CONNECTION_STRING is required.\n"
        "   Set it in your .env file."
    )

SERVICE_NAME = "sk-evaluation-pipeline"

configure_azure_monitor(
    connection_string=CONNECTION_STRING,
    logger_name=SERVICE_NAME,
    enable_live_metrics=True,
)

# Enable Semantic Kernel's OpenTelemetry instrumentation
# SK uses the standard OTEL env var to enable detailed tracing
os.environ["SEMANTICKERNEL_EXPERIMENTAL_GENAI_ENABLE_OTEL_DIAGNOSTICS"] = "true"
os.environ["SEMANTICKERNEL_EXPERIMENTAL_GENAI_ENABLE_OTEL_DIAGNOSTICS_SENSITIVE"] = "true"

tracer = trace.get_tracer(SERVICE_NAME, "1.0.0")

print("✅ Azure Monitor OpenTelemetry configured")
print("✅ Semantic Kernel OTEL diagnostics enabled")
print("   All SK traces will be exported to Application Insights")

## Part 2: Semantic Kernel Orchestration Layer

CSV 로그(`query_data_1.csv`)에서 확인되는 실제 운영 파이프라인을 Semantic Kernel으로 재현합니다.

### Pipeline Flow

| Step | Description | SK Component |
|------|-------------|-------------|
| 1 | Intent Classification | Chat Completion + Structured Output |
| 2 | Agent Routing | SK Plugin 선택 (ProductPlugin / RecommendationPlugin) |
| 3 | Function Calling | `function_choice_behavior=auto` 로 메서드 자동 호출 |
| 4 | Answer Generation | Retrieved context 기반 최종 답변 생성 |

### Agent → Method Mapping (from logs)

| Agent | Method | Intent |
|-------|--------|--------|
| `productAgent` | `search_products` | `product_search` |
| `recommendAgent` | `search_recsys` | `recommendation` |
| `policyAgent` | `search_policy` | `policy` |

In [ ]:
# Define the Intent Classification schema (Structured Output)
from pydantic import BaseModel, Field
from enum import Enum


class IntentType(str, Enum):
    """Intent types derived from Application Insights logs."""
    PRODUCT_SEARCH = "product_search"
    RECOMMENDATION = "recommendation"
    POLICY = "policy"
    BEAUTY = "beauty"
    UNKNOWN = "unknown"


class IntentResult(BaseModel):
    """Structured output for intent classification."""
    intent: IntentType = Field(description="Classified intent type")
    confidence: float = Field(
        description="Confidence score between 0 and 1", ge=0.0, le=1.0
    )
    reasoning: str = Field(description="Brief explanation for classification")


# Mapping: intent → expected agent and method (ground truth from logs)
INTENT_AGENT_MAP: Dict[str, Dict[str, str]] = {
    "product_search": {
        "agent": "productAgent",
        "method": "search_products",
    },
    "recommendation": {
        "agent": "recommendAgent",
        "method": "search_recsys",
    },
    "policy": {
        "agent": "policyAgent",
        "method": "search_policy",
    },
    "beauty": {
        "agent": "beautyAgent",
        "method": "search_beauty",
    },
}

print("✅ Intent schema defined")
print(f"   Supported intents: {[e.value for e in IntentType]}")
print(f"   Agent mappings: {list(INTENT_AGENT_MAP.keys())}")

In [ ]:
# Define Semantic Kernel Plugins (Agent wrappers)
import semantic_kernel as sk
from semantic_kernel.connectors.ai.open_ai import AzureChatCompletion
from semantic_kernel.functions import kernel_function


class ProductPlugin:
    """SK Plugin wrapping productAgent - product search functionality."""

    @kernel_function(
        name="search_products",
        description="Search for product information by query"
    )
    def search_products(self, query: str) -> str:
        """
        Search products based on user query.

        In production, this calls the actual product search MCP/API.
        Here we simulate the retrieval that produces AGENT_OUTPUT_RAW logs.
        """
        with tracer.start_as_current_span("productAgent.search_products") as span:
            span.set_attribute("agent.name", "productAgent")
            span.set_attribute("agent.method", "search_products")
            span.set_attribute("agent.query", query)
            # Simulated product retrieval context
            return (
                "검색된 제품 정보: 마몽드 플래시토닝 데이지 리퀴드 마스크 - "
                "중성~복합성 피부에 최적화, 10% PHA 함유, "
                "모공 정돈 및 피부결 개선 효과. 가격: 30,000원. "
                "리뷰 평점: 4.8/5.0 (411건)"
            )


class RecommendationPlugin:
    """SK Plugin wrapping recommendAgent - recommendation functionality."""

    @kernel_function(
        name="search_recommendations",
        description="Search for personalized product recommendations"
    )
    def search_recommendations(self, query: str) -> str:
        """
        Get personalized recommendations based on user query.

        In production, this calls the recommendation engine (recsys).
        """
        with tracer.start_as_current_span("recommendAgent.search_recsys") as span:
            span.set_attribute("agent.name", "recommendAgent")
            span.set_attribute("agent.method", "search_recsys")
            span.set_attribute("agent.query", query)
            return (
                "추천 제품: 1) 이니스프리 수퍼 화산송이 모공 마스크 - "
                "지성/복합성 피부용, 가격 15,000원. "
                "2) 코스알엑스 펩타이드 콜라겐 아이패치 - "
                "중성/건성 피부용, 가격 22,000원."
            )


print("✅ SK Plugins defined: ProductPlugin, RecommendationPlugin")

In [ ]:
# Build the SK Orchestration Kernel
from semantic_kernel.connectors.ai.function_choice_behavior import (
    FunctionChoiceBehavior,
)
from semantic_kernel.connectors.ai.open_ai import (
    AzureChatPromptExecutionSettings,
)
from semantic_kernel.contents import ChatHistory

# Initialize Kernel
kernel = sk.Kernel()

# Add Azure OpenAI Chat Completion service
chat_service = AzureChatCompletion(
    deployment_name=AZURE_OPENAI_CHAT_DEPLOYMENT_NAME,
    endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    api_version=AZURE_OPENAI_API_VERSION,
)
kernel.add_service(chat_service)

# Register plugins
kernel.add_plugin(ProductPlugin(), plugin_name="ProductPlugin")
kernel.add_plugin(RecommendationPlugin(), plugin_name="RecommendationPlugin")

print("✅ Semantic Kernel initialized")
print(f"   Service: AzureChatCompletion ({AZURE_OPENAI_CHAT_DEPLOYMENT_NAME})")
print("   Plugins: ProductPlugin, RecommendationPlugin")

In [ ]:
# Orchestration: Intent Classification → Agent Routing → Answer Generation
async def classify_intent(query: str) -> IntentResult:
    """Step 1: Classify user intent using structured output."""
    with tracer.start_as_current_span("intent_classification") as span:
        span.set_attribute("user.query", query)

        chat = ChatHistory()
        chat.add_system_message(
            "You are an intent classifier for a beauty/cosmetics chat app. "
            "Classify the user query into one of these intents:\n"
            "- product_search: questions about specific products\n"
            "- recommendation: requests for product recommendations\n"
            "- policy: questions about policies (return, shipping, etc.)\n"
            "- beauty: general beauty tips and advice\n"
            "- unknown: cannot classify\n\n"
            "Respond ONLY with valid JSON matching this schema:\n"
            '{"intent": "<type>", "confidence": <0-1>, "reasoning": "<why>"}'
        )
        chat.add_user_message(query)

        settings = AzureChatPromptExecutionSettings(
            temperature=0.0,
            response_format={"type": "json_object"},
        )

        response = await kernel.invoke_prompt(
            prompt="{{$chat_history}}",
            chat_history=chat,
            settings=settings,
        )

        try:
            result_dict = json.loads(str(response))
            intent_result = IntentResult(**result_dict)
        except (json.JSONDecodeError, Exception):
            intent_result = IntentResult(
                intent=IntentType.UNKNOWN,
                confidence=0.0,
                reasoning="Failed to parse intent",
            )

        span.set_attribute("intent.type", intent_result.intent.value)
        span.set_attribute("intent.confidence", intent_result.confidence)
        return intent_result


async def route_and_execute(query: str, intent: IntentResult) -> str:
    """Step 2-3: Route to agent plugin and execute via function calling auto."""
    with tracer.start_as_current_span("agent_routing") as span:
        span.set_attribute("intent.type", intent.intent.value)

        chat = ChatHistory()
        chat.add_system_message(
            "You are a beauty product assistant. "
            "Use the available tools to find relevant information, "
            "then provide a helpful answer to the user."
        )
        chat.add_user_message(query)

        # Enable auto function calling - SK will choose the right plugin
        settings = AzureChatPromptExecutionSettings(
            function_choice_behavior=FunctionChoiceBehavior.Auto(
                auto_invoke=True
            ),
        )

        response = await kernel.invoke_prompt(
            prompt="{{$chat_history}}",
            chat_history=chat,
            settings=settings,
        )

        span.set_attribute("response.length", len(str(response)))
        return str(response)


async def run_pipeline(query: str) -> Dict[str, Any]:
    """Run the full orchestration pipeline."""
    with tracer.start_as_current_span("evaluation_pipeline") as span:
        span.set_attribute("user.query", query)

        # Step 1: Intent Classification
        intent = await classify_intent(query)

        # Step 2-3: Agent Routing + Function Calling
        answer = await route_and_execute(query, intent)

        result = {
            "query": query,
            "intent": intent.intent.value,
            "confidence": intent.confidence,
            "answer": answer,
        }

        span.set_attribute("pipeline.intent", intent.intent.value)
        return result

print("✅ Orchestration functions defined")
print("   classify_intent() → route_and_execute() → run_pipeline()")

In [ ]:
# Test the pipeline with a sample query
test_query = "마몽드 플래시토닝 리퀴드 마스크는 어떤 피부 타입에 좋아?"

result = await run_pipeline(test_query)

print("=" * 60)
print("🧪 Pipeline Test Result")
print("=" * 60)
print(f"Query: {result['query']}")
print(f"Intent: {result['intent']} (confidence: {result['confidence']:.2f})")
print(f"Answer: {result['answer'][:200]}...")

## Part 3: Build Evaluation Dataset from App Insights Logs

`query_data_1.csv` 파일을 파싱하여 트레이스 단위로 그룹핑하고, 각 구간별 평가에 필요한 데이터셋을 구성합니다.

### Log Event Types (custom_type)

| custom_type | Description | Evaluation Use |
|-------------|-------------|----------------|
| `USER_QUERY` | 사용자 질의 | 평가 입력(query) |
| `AGENT_NAME` | 호출된 에이전트.메서드 | Agent/Method 정답 |
| `AGENT_OUTPUT_RAW` | 에이전트 검색 결과 | Context (groundedness) |
| `FINAL_ANSWER_RAW` | 최종 생성 답변 | Response (similarity) |

In [ ]:
# Parse Application Insights CSV logs into evaluation records
@dataclass
class EvalRecord:
    """A single evaluation record reconstructed from trace logs."""
    trace_id: str
    session_id: str
    user_query: str
    agent_name: str           # e.g., "productAgent.search_products"
    expected_intent: str      # derived: "product_search"
    expected_agent: str       # derived: "productAgent"
    expected_method: str      # derived: "search_products"
    agent_context: str        # from AGENT_OUTPUT_RAW
    final_answer: str         # from FINAL_ANSWER_RAW


def derive_intent_from_agent(agent_name: str) -> tuple[str, str, str]:
    """Derive intent, agent, method from AGENT_NAME log value."""
    # agent_name format: "productAgent.search_products"
    parts = agent_name.split(".")
    agent = parts[0] if len(parts) > 0 else "unknown"
    method = parts[1] if len(parts) > 1 else "unknown"

    intent_map = {
        "productAgent": "product_search",
        "recommendAgent": "recommendation",
        "policyAgent": "policy",
        "beautyAgent": "beauty",
        "myskinAgent": "beauty",
        "btsAgent": "unknown",
    }
    intent = intent_map.get(agent, "unknown")
    return intent, agent, method


def parse_csv_to_eval_records(
    csv_path: str, max_records: int = 50
) -> List[EvalRecord]:
    """
    Parse Application Insights CSV export into evaluation records.

    Groups log entries by trace_id to reconstruct complete request flows.
    """
    # Phase 1: Group rows by trace_id
    trace_groups: Dict[str, Dict[str, str]] = {}

    with open(csv_path, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            try:
                dims = json.loads(row.get('customDimensions', '{}'))
            except json.JSONDecodeError:
                continue

            custom_type = dims.get('custom_type', '')
            trace_id = dims.get('dd.trace_id', '')
            session_id = dims.get('session_id', '')

            if not trace_id:
                continue

            if trace_id not in trace_groups:
                trace_groups[trace_id] = {
                    "session_id": session_id,
                }

            group = trace_groups[trace_id]
            message = row.get('message', '')

            if custom_type == 'USER_QUERY':
                group['user_query'] = message
            elif custom_type == 'AGENT_NAME':
                group['agent_name'] = message
            elif custom_type == 'AGENT_OUTPUT_RAW':
                group['agent_context'] = message[:2000]
            elif custom_type == 'FINAL_ANSWER_RAW':
                group['final_answer'] = message[:2000]

    # Phase 2: Build EvalRecords from complete traces
    records: List[EvalRecord] = []
    for trace_id, group in trace_groups.items():
        if not all(
            k in group
            for k in ['user_query', 'agent_name', 'final_answer']
        ):
            continue

        intent, agent, method = derive_intent_from_agent(
            group['agent_name']
        )
        records.append(EvalRecord(
            trace_id=trace_id,
            session_id=group.get('session_id', ''),
            user_query=group['user_query'],
            agent_name=group['agent_name'],
            expected_intent=intent,
            expected_agent=agent,
            expected_method=method,
            agent_context=group.get('agent_context', ''),
            final_answer=group.get('final_answer', ''),
        ))

        if len(records) >= max_records:
            break

    return records


print("✅ CSV parser defined")

In [ ]:
# Load and parse the Application Insights log data
from collections import Counter

CSV_PATH = "./log/query_data_1.csv"
eval_records = parse_csv_to_eval_records(CSV_PATH, max_records=50)

print(f"✅ Loaded {len(eval_records)} evaluation records from CSV")
print("\nIntent distribution:")
intent_counts = Counter(r.expected_intent for r in eval_records)
for intent, count in intent_counts.most_common():
    print(f"   {intent}: {count}")

print("\nAgent distribution:")
agent_counts = Counter(r.expected_agent for r in eval_records)
for agent, count in agent_counts.most_common():
    print(f"   {agent}: {count}")

# Show sample record
if eval_records:
    sample = eval_records[0]
    print("\n--- Sample Record ---")
    print(f"   Query: {sample.user_query[:80]}...")
    print(f"   Intent: {sample.expected_intent}")
    print(f"   Agent: {sample.expected_agent}.{sample.expected_method}")

In [ ]:
# Run SK pipeline on evaluation queries to get predicted values

async def run_pipeline_for_eval(records: List[EvalRecord]) -> List[Dict]:
    """Run the SK pipeline on each record and collect predictions."""
    eval_data = []
    for i, record in enumerate(records):
        try:
            # Classify intent using SK
            intent = await classify_intent(record.user_query)
            predicted_intent = intent.intent.value

            # Determine predicted agent/method from intent
            mapping = INTENT_AGENT_MAP.get(predicted_intent, {})
            predicted_agent = mapping.get("agent", "unknown")
            predicted_method = mapping.get("method", "unknown")
        except Exception:
            predicted_intent = "unknown"
            predicted_agent = "unknown"
            predicted_method = "unknown"

        eval_data.append({
            "query": record.user_query,
            # Ground truth (from logs)
            "expected_intent": record.expected_intent,
            "expected_agent": record.expected_agent,
            "expected_method": record.expected_method,
            "context": record.agent_context,
            "ground_truth": record.final_answer,
            # Predictions (from SK pipeline)
            "predicted_intent": predicted_intent,
            "predicted_agent": predicted_agent,
            "predicted_method": predicted_method,
            "response": record.final_answer,  # For pre-built evaluators
        })

        if (i + 1) % 10 == 0:
            print(f"   Processed {i + 1}/{len(records)} records...")

    return eval_data

# Use a subset for faster evaluation
eval_subset = eval_records[:20]
print(f"🔄 Running SK pipeline on {len(eval_subset)} records...")
eval_data = await run_pipeline_for_eval(eval_subset)
print(f"✅ Evaluation dataset ready: {len(eval_data)} records")

# Save as JSONL for the evaluate() API
EVAL_DATA_PATH = "./log/eval_dataset.jsonl"
with open(EVAL_DATA_PATH, 'w', encoding='utf-8') as f:
    for item in eval_data:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')
print(f"💾 Saved to {EVAL_DATA_PATH}")

## Part 4: Evaluation Step 1 – Intent Classification (Exact Match)

첫 번째 평가 구간: **Intent Classification**의 정확도를 측정합니다.

SK가 분류한 intent와 실제 로그에서 기록된 intent(AGENT_NAME에서 추출)를 **exact matching**으로 비교합니다.

| Metric | Description |
|--------|-------------|
| `intent_accuracy` | 1.0 if predicted == expected, else 0.0 |
| `intent_match` | "PASS" or "FAIL" |

In [ ]:
# Custom Evaluator: Intent Classification Accuracy (Exact Match)
from azure.ai.evaluation import evaluate


class IntentAccuracyEvaluator:
    """
    Custom evaluator for intent classification accuracy.

    Compares predicted intent against expected intent using exact matching.
    Returns accuracy score (1.0 or 0.0) and pass/fail label.
    """

    def __init__(self):
        self._name = "intent_accuracy"

    def __call__(
        self,
        *,
        predicted_intent: str,
        expected_intent: str,
        **kwargs,
    ) -> Dict[str, Any]:
        is_match = predicted_intent.strip().lower() == (
            expected_intent.strip().lower()
        )
        return {
            "intent_accuracy": 1.0 if is_match else 0.0,
            "intent_match": "PASS" if is_match else "FAIL",
            "predicted": predicted_intent,
            "expected": expected_intent,
        }


# Run intent classification evaluation
intent_evaluator = IntentAccuracyEvaluator()

intent_result = evaluate(
    data=EVAL_DATA_PATH,
    evaluators={"intent_accuracy": intent_evaluator},
    evaluator_config={
        "intent_accuracy": {
            "column_mapping": {
                "predicted_intent": "${data.predicted_intent}",
                "expected_intent": "${data.expected_intent}",
            }
        }
    },
    output_path="./log/eval_step1_intent.json",
)

print("=" * 60)
print("📊 Step 1: Intent Classification Evaluation")
print("=" * 60)
metrics = intent_result.get("metrics", {})
for k, v in metrics.items():
    print(f"   {k}: {v}")

## Part 5: Evaluation Step 2 – Agent Relevance

두 번째 평가 구간: intent에 따라 **올바른 에이전트가 호출되었는지** 평가합니다.

예를 들어, `product_search` intent에 대해 `productAgent`가 호출되어야 하며,
`recommendation` intent에는 `recommendAgent`가 호출되어야 합니다.

In [ ]:
# Custom Evaluator: Agent Relevance
class AgentRelevanceEvaluator:
    """
    Custom evaluator for agent routing accuracy.

    Checks whether the predicted agent matches the expected agent
    based on the intent-to-agent mapping.
    """

    def __init__(self, intent_agent_map: Dict[str, Dict[str, str]]):
        self._intent_agent_map = intent_agent_map

    def __call__(
        self,
        *,
        predicted_agent: str,
        expected_agent: str,
        predicted_intent: str,
        **kwargs,
    ) -> Dict[str, Any]:
        # Check direct match
        is_match = predicted_agent.strip().lower() == (
            expected_agent.strip().lower()
        )

        # Also verify the intent→agent mapping is correct
        expected_from_map = self._intent_agent_map.get(
            predicted_intent, {}
        ).get("agent", "")
        mapping_consistent = expected_from_map.lower() == (
            predicted_agent.lower()
        )

        return {
            "agent_relevance": 1.0 if is_match else 0.0,
            "agent_match": "PASS" if is_match else "FAIL",
            "mapping_consistent": mapping_consistent,
            "predicted_agent": predicted_agent,
            "expected_agent": expected_agent,
        }


# Run agent relevance evaluation
agent_evaluator = AgentRelevanceEvaluator(INTENT_AGENT_MAP)

agent_result = evaluate(
    data=EVAL_DATA_PATH,
    evaluators={"agent_relevance": agent_evaluator},
    evaluator_config={
        "agent_relevance": {
            "column_mapping": {
                "predicted_agent": "${data.predicted_agent}",
                "expected_agent": "${data.expected_agent}",
                "predicted_intent": "${data.predicted_intent}",
            }
        }
    },
    output_path="./log/eval_step2_agent.json",
)

print("=" * 60)
print("📊 Step 2: Agent Relevance Evaluation")
print("=" * 60)
metrics = agent_result.get("metrics", {})
for k, v in metrics.items():
    print(f"   {k}: {v}")

## Part 6: Evaluation Step 3 – Method Relevance

세 번째 평가 구간: function calling을 통해 **올바른 메서드가 호출되었는지** 평가합니다.

에이전트가 올바르게 선택되더라도, 해당 에이전트 내의 메서드(`search_products`, `search_recsys` 등)가 적절한지를 추가로 검증합니다.

In [ ]:
# Custom Evaluator: Method Relevance
class MethodRelevanceEvaluator:
    """
    Custom evaluator for function calling method relevance.

    Verifies that the correct method was called within the selected agent.
    Checks both exact match and intent-to-method mapping consistency.
    """

    def __init__(self, intent_agent_map: Dict[str, Dict[str, str]]):
        self._intent_agent_map = intent_agent_map

    def __call__(
        self,
        *,
        predicted_method: str,
        expected_method: str,
        predicted_intent: str,
        **kwargs,
    ) -> Dict[str, Any]:
        is_match = predicted_method.strip().lower() == (
            expected_method.strip().lower()
        )

        # Verify method matches the intent mapping
        expected_from_map = self._intent_agent_map.get(
            predicted_intent, {}
        ).get("method", "")
        mapping_consistent = expected_from_map.lower() == (
            predicted_method.lower()
        )

        return {
            "method_relevance": 1.0 if is_match else 0.0,
            "method_match": "PASS" if is_match else "FAIL",
            "mapping_consistent": mapping_consistent,
            "predicted_method": predicted_method,
            "expected_method": expected_method,
        }


# Run method relevance evaluation
method_evaluator = MethodRelevanceEvaluator(INTENT_AGENT_MAP)

method_result = evaluate(
    data=EVAL_DATA_PATH,
    evaluators={"method_relevance": method_evaluator},
    evaluator_config={
        "method_relevance": {
            "column_mapping": {
                "predicted_method": "${data.predicted_method}",
                "expected_method": "${data.expected_method}",
                "predicted_intent": "${data.predicted_intent}",
            }
        }
    },
    output_path="./log/eval_step3_method.json",
)

print("=" * 60)
print("📊 Step 3: Method Relevance Evaluation")
print("=" * 60)
metrics = method_result.get("metrics", {})
for k, v in metrics.items():
    print(f"   {k}: {v}")

## Part 7: Evaluation Step 4 – Similarity & Groundedness (Pre-built)

네 번째 평가 구간: 최종 생성 답변의 품질을 **Pre-built Evaluator**로 평가합니다.

| Evaluator | Description | Score Range |
|-----------|-------------|-------------|
| `SimilarityEvaluator` | 답변과 ground truth 간 의미적 유사도 | 1-5 |
| `GroundednessEvaluator` | 답변이 context에 근거하는 정도 | 1-5 |

Azure AI Evaluation SDK의 `evaluate()` 함수를 사용하며, Foundry 프로젝트에 결과가 자동으로 기록됩니다.

In [ ]:
# Pre-built Evaluators: Similarity & Groundedness
from azure.ai.evaluation import (
    SimilarityEvaluator,
    GroundednessEvaluator,
    AzureOpenAIModelConfiguration,
)

# Model config for AI-assisted evaluators
model_config = AzureOpenAIModelConfiguration(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    azure_deployment=AZURE_OPENAI_CHAT_DEPLOYMENT_NAME,
    api_version=AZURE_OPENAI_API_VERSION,
)

# Initialize pre-built evaluators
similarity_eval = SimilarityEvaluator(model_config=model_config)
groundedness_eval = GroundednessEvaluator(model_config=model_config)

print("✅ Pre-built evaluators initialized")
print("   - SimilarityEvaluator (query + response + ground_truth)")
print("   - GroundednessEvaluator (query + response + context)")

In [ ]:
# Run Similarity & Groundedness evaluation
quality_result = evaluate(
    data=EVAL_DATA_PATH,
    evaluators={
        "similarity": similarity_eval,
        "groundedness": groundedness_eval,
    },
    evaluator_config={
        "similarity": {
            "column_mapping": {
                "query": "${data.query}",
                "response": "${data.response}",
                "ground_truth": "${data.ground_truth}",
            }
        },
        "groundedness": {
            "column_mapping": {
                "query": "${data.query}",
                "response": "${data.response}",
                "context": "${data.context}",
            }
        },
    },
    azure_ai_project=AZURE_AI_PROJECT_ENDPOINT,
    output_path="./log/eval_step4_quality.json",
)

print("=" * 60)
print("📊 Step 4: Similarity & Groundedness Evaluation")
print("=" * 60)
metrics = quality_result.get("metrics", {})
for k, v in metrics.items():
    print(f"   {k}: {v}")

## Part 8: Evaluation Summary Dashboard

모든 구간별 평가 결과를 통합하여 **Evaluation Summary**를 출력합니다.

```
┌─────────────────────────────────────────────────────────────────┐
│                    Evaluation Pipeline                           │
│                                                                  │
│  Step 1        Step 2         Step 3         Step 4              │
│  Intent ──► Agent ──► Method ──► Answer Quality                  │
│  (Exact       (Custom        (Custom        (Similarity +        │
│   Match)       Eval)          Eval)          Groundedness)        │
│                                                                  │
│  Each step evaluates a specific stage of the SK pipeline         │
│  using logs from Application Insights as ground truth            │
└─────────────────────────────────────────────────────────────────┘
```

In [ ]:
# Aggregate all evaluation results into a summary
def extract_metric(result: dict, prefix: str) -> Dict[str, Any]:
    """Extract metrics with a specific prefix from evaluation result."""
    metrics = result.get("metrics", {})
    return {k: v for k, v in metrics.items() if prefix in k.lower()}


summary = {
    "Step 1: Intent Classification": extract_metric(
        intent_result, "intent"
    ),
    "Step 2: Agent Relevance": extract_metric(
        agent_result, "agent"
    ),
    "Step 3: Method Relevance": extract_metric(
        method_result, "method"
    ),
    "Step 4: Similarity": extract_metric(
        quality_result, "similarity"
    ),
    "Step 4: Groundedness": extract_metric(
        quality_result, "groundedness"
    ),
}

print("=" * 70)
print("📊 EVALUATION PIPELINE SUMMARY")
print("=" * 70)

for step_name, metrics in summary.items():
    print(f"\n🔹 {step_name}")
    if metrics:
        for k, v in metrics.items():
            if isinstance(v, float):
                print(f"     {k}: {v:.4f}")
            else:
                print(f"     {k}: {v}")
    else:
        print("     (no metrics)")

print("\n" + "=" * 70)
print("📁 Detailed results saved to ./log/eval_step*.json")
print(f"🌐 View in Foundry Portal: {AZURE_AI_PROJECT_ENDPOINT}")

In [ ]:
# Row-level result inspection
print("=" * 70)
print("📋 Row-Level Results (first 5 records)")
print("=" * 70)

rows = intent_result.get("rows", [])
for i, row in enumerate(rows[:5]):
    print(f"\n[{i+1}] Query: {row.get('inputs.query', 'N/A')[:60]}...")
    print(f"    Intent: expected={row.get('inputs.expected_intent', '?')}"
          f" / predicted={row.get('inputs.predicted_intent', '?')}"
          f" → {row.get('outputs.intent_accuracy.intent_match', '?')}")

agent_rows = agent_result.get("rows", [])
for i, row in enumerate(agent_rows[:5]):
    print(f"\n[{i+1}] Agent: expected={row.get('inputs.expected_agent', '?')}"
          f" / predicted={row.get('inputs.predicted_agent', '?')}"
          f" → {row.get('outputs.agent_relevance.agent_match', '?')}")

## Wrap-up

### Evaluation Pipeline 요약

| Step | Evaluator Type | Metric | Description |
|------|---------------|--------|-------------|
| 1 | Custom (Exact Match) | `intent_accuracy` | Intent 분류 정확도 |
| 2 | Custom | `agent_relevance` | Agent 라우팅 정확도 |
| 3 | Custom | `method_relevance` | Function calling 메서드 정확도 |
| 4 | Pre-built | `similarity` | 답변-정답 의미 유사도 (1-5) |
| 4 | Pre-built | `groundedness` | 답변-context 근거성 (1-5) |

### Key Takeaways

- Application Insights 로그를 파싱하여 구간별 ground truth를 추출할 수 있습니다
- Custom evaluator는 Python callable로 간단히 구현하여 `evaluate()`에 전달합니다
- Pre-built evaluator(`SimilarityEvaluator`, `GroundednessEvaluator`)는 LLM judge 기반으로 동작합니다
- 모든 결과는 Foundry 프로젝트에 자동 기록되어 Portal에서 확인할 수 있습니다

### Next Steps

1. **평가 데이터 확장**: `max_records`를 늘려 전체 로그에 대해 평가를 수행하세요
2. **Evals API 활용**: `4_agent_fleet_management.ipynb`의 Evals API 패턴으로 클라우드 평가를 실행하세요
3. **CI/CD 통합**: 평가 파이프라인을 자동화하여 모델 업데이트 시 회귀 테스트로 사용하세요
4. **추가 Evaluator**: `RelevanceEvaluator`, `CoherenceEvaluator` 등을 추가하여 더 다양한 품질 지표를 수집하세요

## Additional Resources

- [Azure AI Evaluation SDK Documentation](https://learn.microsoft.com/en-us/azure/ai-foundry/how-to/develop/evaluate-sdk)
- [Custom Evaluators Guide](https://learn.microsoft.com/en-us/azure/ai-foundry/concepts/evaluation-evaluators/custom-evaluators)
- [Semantic Kernel OpenTelemetry](https://learn.microsoft.com/en-us/semantic-kernel/concepts/enterprise-readiness/observability)
- [Application Insights for AI Apps](https://learn.microsoft.com/en-us/azure/azure-monitor/app/app-insights-overview)
- [Azure AI Foundry Portal](https://ai.azure.com/)